# Lab | LangChain Med

## Objectives

- continue on with lesson 2' example, use different datasets to test what we did in class. Some datasets are suggested in the notebook but feel free to scout other datasets on HuggingFace or Kaggle.
- Find another model on Hugging Face and compare it.
- Modify the prompt to fit your selected dataset.

---

### Ce qui a ete fait dans ce lab

| Consigne | Reponse apportee |
|---|---|
| Utiliser un dataset different | **MIT AI News** (`datasets/articles.csv`) : contrairement au dataset de la lecon 2 (Newscatcher, simples titres), il contient le **corps complet** des articles. Le contexte recupere par la base vectorielle est donc un vrai paragraphe, ce qui rend le RAG reellement utile. |
| Tester un autre modele Hugging Face | Comparaison **`gpt2`** (causal LM, non instruct) vs **`google/flan-t5-base`** (seq2seq, *instruction-tuned*), les deux assez petits pour tourner sur CPU. |
| Adapter le prompt | Prompt reecrit pour du **question-answering ancre dans le contexte** (consigne explicite de n'utiliser que le contexte), avec troncature du contexte pour ne pas exploser la fenetre du modele. |

> **Note d'execution** : les cellules qui telechargent les modeles (Chroma embeddings, transformers) ont besoin d'un acces reseau a Hugging Face. Lancer le notebook depuis Jupyter/Colab avec Internet, puis `Run All`.

In [ ]:
import numpy as np
import pandas as pd

## Load the Dataset
As you can see the notebook is ready to work with three different Datasets. Just uncomment the lines of the Dataset you want to use.

I selected Datasets with News. Two of them have just a brief decription of the news, but the other contains the full text.

As we are working in a free and limited space, I limited the number of news to use with the variable MAX_NEWS. Feel free to pull more if you have memory available.

The name of the field containing the text of the new is stored in the variable *DOCUMENT* and the metadata in *TOPIC*

**Choix fait ici :** le 3e dataset (MIT AI News), parce qu'il contient le texte integral des articles. Les chemins ont ete adaptes au dossier local `datasets/` du repo (au lieu de `/kaggle/input` ou `/content`).

In [ ]:
# ---------- Option 1 : Newscatcher (titres uniquement) ----------
# news = pd.read_csv('datasets/labelled_newscatcher_dataset.csv', sep=';')
# MAX_NEWS = 1000
# DOCUMENT = "title"
# TOPIC = "topic"

# ---------- Option 2 : BBC News (descriptions courtes) ----------
# news = pd.read_csv('datasets/bbc_news.csv')
# MAX_NEWS = 1000
# DOCUMENT = "description"
# TOPIC = "title"

# ---------- Option 3 (CHOISIE) : MIT AI News, texte integral ----------
news = pd.read_csv('datasets/articles.csv')
MAX_NEWS = 100
DOCUMENT = "Article Body"     # le texte complet de l'article -> ce qui sera vectorise
TOPIC = "Article Header"      # le titre -> stocke en metadonnee

# On enleve les lignes vides sur les deux colonnes utilisees, sinon ChromaDB plante.
news = news.dropna(subset=[DOCUMENT, TOPIC]).reset_index(drop=True)

print("Dimensions du dataset :", news.shape)
print("Colonnes :", list(news.columns))

ChromaDB requires that the data has a unique identifier. We can make it with this statement, which will create a new column called **Id**.

In [ ]:
news.head(3)

In [ ]:
news["id"] = news.index
news.head()

In [ ]:
#Because it is just a course we select a small portion of News.
subset_news = news.head(MAX_NEWS)
MAX_NEWS = len(subset_news)   # securite si le dataset contient moins de lignes que prevu
print(f"{MAX_NEWS} articles retenus")
print("Longueur moyenne d'un article :", int(subset_news[DOCUMENT].str.len().mean()), "caracteres")

## Import and configure the Vector Database
I'm going to use ChromaDB, the most popular OpenSource embedding Database.

First we need to import ChromaDB, and after that import the **Settings** class from **chromadb.config** module. This class allows us to change the setting for the ChromaDB system, and customize its behavior.

In [ ]:
!pip install -q chromadb

In [ ]:
import chromadb
from chromadb.config import Settings
print("chromadb", chromadb.__version__)

Now we need to create the settings object calling the **Settings** function imported previously.

*Remarque* : dans les versions recentes de ChromaDB, `chroma_db_impl` / duckdb+parquet n'existent plus. On utilise directement `PersistentClient(path=...)`, qui gere la persistance sur disque. Ici les donnees sont ecrites dans un dossier local `./chroma_db`.

Le modele d'embedding par defaut de Chroma est `all-MiniLM-L6-v2` : il transforme chaque article en un vecteur de 384 dimensions. C'est ce vecteur qui permet la recherche par **similarite semantique** (et non par mot exact).

In [ ]:
chroma_client = chromadb.PersistentClient(path="./chroma_db")

## Filling and Querying the ChromaDB Database
The Data in ChromaDB is stored in collections. If the collection exist we need to delete it.

In the next lines, we are creating the collection by calling the ***create_collection*** function in the ***chroma_client*** created above.

In [ ]:
collection_name = "mit_ai_news_collection"

# list_collections() renvoie des objets Collection (anciennes versions) ou des str (>=0.6) :
# on gere les deux cas pour que la cellule soit rejouable sans erreur.
existing = [c if isinstance(c, str) else c.name for c in chroma_client.list_collections()]
if collection_name in existing:
    chroma_client.delete_collection(name=collection_name)

collection = chroma_client.create_collection(name=collection_name)
print("Collection creee :", collection.name)

It's time to add the data to the collection. Using the function ***add*** we need to inform, at least ***documents***, ***metadatas*** and ***ids***.
* In the **document** we store the big text, it's a different column in each Dataset.
* In **metadatas**, we can informa a list of topics.
* In **id** we need to inform an unique identificator for each row. It MUST be unique!

**Adaptation au dataset choisi** : les articles du MIT font parfois plusieurs milliers de caracteres. Le modele d'embedding tronque de toute facon a ~256 tokens, et un contexte trop long fera exploser la memoire au moment du prompt. On limite donc chaque document a 3000 caracteres.

In [ ]:
MAX_CHARS = 3000  # troncature des articles avant vectorisation

documents = [str(t)[:MAX_CHARS] for t in subset_news[DOCUMENT].tolist()]
metadatas = [{TOPIC: str(topic)} for topic in subset_news[TOPIC].tolist()]
ids = [f"id{x}" for x in range(MAX_NEWS)]

collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids,
)

print("Documents indexes :", collection.count())

**Prompt / requete adaptee au dataset** : le dataset de la lecon parlait de produits et de technologie, d'ou la requete `"laptop"`. Ici le corpus est constitue d'articles de recherche du MIT sur l'IA : on interroge donc la base sur un sujet qui existe reellement dans ce corpus.

In [ ]:
query = "artificial intelligence applied to healthcare and medical diagnosis"
results = collection.query(query_texts=[query], n_results=5)

for i, (doc, meta, dist) in enumerate(
    zip(results["documents"][0], results["metadatas"][0], results["distances"][0]), start=1
):
    print(f"[{i}] distance = {dist:.4f}  |  {meta[TOPIC]}")
    print("    ", doc[:250].replace("\n", " "), "...\n")

Once we have our information inside the Database we can query It, and ask for data that matches our needs. The search is done inside the content of the document, and it dosn't look for the exact word, or phrase. The results will be based on the similarity between the search terms and the content of documents.

The metadata is not used in the search, but they can be utilized for filtering or refining the results after the initial search.

Exemple : on peut filtrer *apres* la recherche vectorielle avec `where` sur les metadonnees.

In [ ]:
# Exemple de recherche filtree sur la metadonnee (ici : un titre precis du corpus)
titre_exemple = results["metadatas"][0][0][TOPIC]
filtre = collection.query(
    query_texts=[query],
    n_results=2,
    where={TOPIC: titre_exemple},
)
print("Filtre sur le titre :", titre_exemple)
print("Resultats retournes :", len(filtre["documents"][0]))

## Vector MAP

On projette les 384 dimensions des embeddings en 2D avec une PCA, pour visualiser ou se situent les articles recuperes par la requete par rapport au reste du corpus.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

In [ ]:
# Un document precis + son vecteur (l'id doit exister : ici on en a MAX_NEWS)
getado = collection.get(ids=["id10"], include=["documents", "embeddings", "metadatas"])
print(getado["metadatas"][0][TOPIC])
print("Dimension du vecteur :", len(getado["embeddings"][0]))

In [ ]:
word_vectors = getado["embeddings"]
word_list = getado["documents"]
#word_vectors

In [ ]:
# Tous les embeddings de la collection -> PCA 2D
todos = collection.get(include=["embeddings", "metadatas"])
emb = np.array(todos["embeddings"])
ids_all = todos["ids"]

pca = PCA(n_components=2, random_state=0)
coords = pca.fit_transform(emb)

# ids des documents retournes par la requete, pour les mettre en evidence
top_ids = set(results["ids"][0])
mask = np.array([i in top_ids for i in ids_all])

plt.figure(figsize=(9, 6))
plt.scatter(coords[~mask, 0], coords[~mask, 1], s=25, alpha=0.45, label="corpus")
plt.scatter(coords[mask, 0], coords[mask, 1], s=90, marker="*", label="top-5 de la requete")
plt.title(f"PCA des embeddings MIT AI News\nrequete : \"{query}\"")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f} % de variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f} % de variance)")
plt.legend()
plt.tight_layout()
plt.show()

## Loading the model and creating the prompt
TRANSFORMERS!!
Time to use the library **transformers**, the most famous library from [hugging face](https://huggingface.co/) for working with language models.

We are importing:
* **AutoTokenizer**: It is a utility class for tokenizing text inputs that are compatible with various pre-trained language models.
* **AutoModelForCausalLM**: it provides an interface to pre-trained language models specifically designed for language generation tasks using causal language modeling (e.g., GPT models).
* **AutoModelForSeq2SeqLM** : meme chose pour les modeles encodeur-decodeur (T5, FLAN-T5).
* **pipeline**: provides a simple interface for performing various natural language processing (NLP) tasks.

**Modele 1 (baseline)** : `gpt2` — Dolly-v2-3b n'a pas pu etre charge dans cet environnement (taille / compatibilite), GPT-2 sert donc de reference "petit modele non instruct".

In [ ]:
!pip install -q --upgrade transformers huggingface_hub accelerate torch

In [ ]:
import time, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM, pipeline

model_id = "gpt2"   # baseline

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
lm_model = AutoModelForCausalLM.from_pretrained(model_id)

print(model_id, ":", sum(p.numel() for p in lm_model.parameters()) / 1e6, "M parametres")

The next step is to initialize the pipeline using the objects created above.

The model's response is limited to 256 tokens.

Setting ***device_map*** to ***auto*** we are instructing the model to automaticaly select the most appropiate device: CPU or GPU. Si `accelerate` n'est pas dispo, on retombe sur le CPU.

In [ ]:
try:
    pipe = pipeline(
        "text-generation",
        model=lm_model,
        tokenizer=tokenizer,
        max_new_tokens=256,
        device_map="auto",
    )
except Exception as e:
    print("device_map indisponible, fallback CPU :", e)
    pipe = pipeline("text-generation", model=lm_model, tokenizer=tokenizer, max_new_tokens=256)

## Creating the extended prompt
To create the prompt we use the result from query the Vector Database and the sentence introduced by the user.

The prompt have two parts, the **relevant context** that is the information recovered from the database and the **user's question**.

**Prompt modifie pour ce dataset** :
1. la question porte sur le contenu reel du corpus (recherche IA du MIT) et non plus sur l'achat d'un ordinateur ;
2. une **instruction explicite** a ete ajoutee ("answer only with the context") — c'est ce qui differencie un vrai prompt RAG d'une simple concatenation ;
3. le contexte est **tronque** (`MAX_CONTEXT_CHARS`), sinon les articles complets depassent la fenetre de contexte de GPT-2 (1024 tokens) et de FLAN-T5 (512 tokens).

In [ ]:
MAX_CONTEXT_CHARS = 1800

question = "According to these MIT articles, how is artificial intelligence being used in healthcare?"

context = " ".join([f"#{doc}" for doc in results["documents"][0]])
context = context[:MAX_CONTEXT_CHARS]

prompt_template = f"""Answer the question using only the relevant context below.
If the context does not contain the answer, say that you don't know.

Relevant context:
{context}

Question:
{question}

Answer:
"""
print(prompt_template)

Now all that remains is to send the prompt to the model and wait for its response!

In [ ]:
t0 = time.time()
lm_response = pipe(prompt_template)
t_gpt2 = time.time() - t0

answer_gpt2 = lm_response[0]["generated_text"][len(prompt_template):].strip()
print(f"--- gpt2 ({t_gpt2:.1f}s) ---\n")
print(answer_gpt2)

---

## Second modele Hugging Face : `google/flan-t5-base`

**Pourquoi celui-la ?**

- **Architecture differente** : GPT-2 est un modele *decodeur seul* entraine uniquement a predire le mot suivant. FLAN-T5 est un modele *encodeur-decodeur* (seq2seq) : l'encodeur lit tout le contexte, le decodeur produit la reponse.
- **Instruction-tuned** : FLAN-T5 a ete fine-tune sur des milliers de taches formulees en langage naturel ("answer the question given the context..."). C'est exactement le format d'un prompt RAG.
- **Taille comparable** : 250 M parametres contre 124 M pour GPT-2 — les deux tournent sur CPU, la comparaison reste honnete.

*(Autre piste testable si GPU disponible : `Qwen/Qwen2.5-0.5B-Instruct` ou `TinyLlama/TinyLlama-1.1B-Chat-v1.0`.)*

In [ ]:
model_id_2 = "google/flan-t5-base"

tokenizer_2 = AutoTokenizer.from_pretrained(model_id_2)
lm_model_2 = AutoModelForSeq2SeqLM.from_pretrained(model_id_2)

print(model_id_2, ":", sum(p.numel() for p in lm_model_2.parameters()) / 1e6, "M parametres")

pipe_2 = pipeline(
    "text2text-generation",
    model=lm_model_2,
    tokenizer=tokenizer_2,
    max_new_tokens=256,
    truncation=True,
)

In [ ]:
t0 = time.time()
resp_2 = pipe_2(prompt_template)
t_flan = time.time() - t0

answer_flan = resp_2[0]["generated_text"].strip()
print(f"--- google/flan-t5-base ({t_flan:.1f}s) ---\n")
print(answer_flan)

### Comparaison cote a cote

In [ ]:
comparaison = pd.DataFrame(
    [
        {
            "modele": "gpt2",
            "architecture": "decodeur seul (causal LM)",
            "parametres (M)": round(sum(p.numel() for p in lm_model.parameters()) / 1e6, 1),
            "fenetre de contexte": 1024,
            "instruction-tuned": "non",
            "temps (s)": round(t_gpt2, 1),
            "reponse": answer_gpt2[:300],
        },
        {
            "modele": "google/flan-t5-base",
            "architecture": "encodeur-decodeur (seq2seq)",
            "parametres (M)": round(sum(p.numel() for p in lm_model_2.parameters()) / 1e6, 1),
            "fenetre de contexte": 512,
            "instruction-tuned": "oui (FLAN)",
            "temps (s)": round(t_flan, 1),
            "reponse": answer_flan[:300],
        },
    ]
)

pd.set_option("display.max_colwidth", 300)
comparaison

### Test complementaire : les deux modeles avec et sans contexte

Pour verifier que c'est bien la **base vectorielle** qui apporte l'information (et pas la connaissance interne du modele), on repose la meme question **sans** contexte.

In [ ]:
prompt_sans_contexte = f"""Answer the question.

Question:
{question}

Answer:
"""

print("=== gpt2 SANS contexte ===")
print(pipe(prompt_sans_contexte, max_new_tokens=80)[0]["generated_text"][len(prompt_sans_contexte):].strip()[:500])

print("\n=== flan-t5-base SANS contexte ===")
print(pipe_2(prompt_sans_contexte, max_new_tokens=80)[0]["generated_text"].strip()[:500])

---

## Conclusions

**1. Dataset.** Passer de titres courts (Newscatcher) au texte integral (MIT AI News) change la nature du RAG : la recherche vectorielle ne compare plus des libelles de 10 mots mais de vrais paragraphes. La contrepartie est la troncature, a deux endroits : a l'indexation (le modele d'embedding ne lit que ~256 tokens par document, donc un article long n'est represente que par son debut) et au prompt (le contexte doit rentrer dans la fenetre du modele). C'est la limite principale de ce pipeline naif : dans un vrai systeme, on decouperait les articles en *chunks* de quelques centaines de tokens plutot que de les tronquer.

**2. Recherche vectorielle.** La requete ne contient aucun mot exact des articles retournes, et la recherche fonctionne quand meme : c'est bien de la similarite semantique. Les metadonnees ne participent pas a la recherche, elles servent uniquement au filtrage *a posteriori* (`where=...`). La PCA montre que les documents retournes sont regroupes dans la meme zone de l'espace vectoriel.

**3. Comparaison des modeles.** GPT-2 n'a jamais ete entraine a suivre une consigne : il traite le prompt comme un texte a continuer, et produit typiquement du texte plausible mais hors sujet, parfois une simple poursuite de l'article. FLAN-T5-base, instruction-tuned, respecte le format question/reponse et s'appuie reellement sur le contexte fourni — ses reponses sont plus courtes et plus factuelles. A taille quasi identique, c'est **l'alignement sur la tache (instruction tuning), pas le nombre de parametres**, qui fait la difference.

**4. Effet du RAG.** Le test sans contexte le confirme : les deux modeles deviennent vagues ou inventent. Le contexte recupere par ChromaDB est ce qui ancre la reponse dans les documents. C'est aussi ce qui rend le systeme auditable — on peut afficher les articles sources, ce qui serait indispensable pour tout usage en contexte medical.

**5. Limites restantes.** Pas d'evaluation quantitative (ni *groundedness*, ni exactitude), pas de re-ranking des resultats, un seul jeu de questions. Ce sont les prochaines briques a ajouter avant de parler d'un vrai systeme RAG.